# Anthropic Agent

### State Diagram (Agent View)
```mermaid
stateDiagram-v2
direction TB
    INIT --> IS_TOOL_CALL
    IS_TOOL_CALL --> CHAT: condition_false
    IS_TOOL_CALL --> TOOL_USE: condition_true
    
    CHAT --> IS_TERMINATE
    TOOL_USE --> IS_TERMINATE
    
    IS_TERMINATE --> IS_TOOL_CALL: condition_false
    IS_TERMINATE --> FINAL: condition_true
```



## Setup: Create Agent

In [1]:
import os
os.environ["LOG_LEVEL"] = "WARNING"

In [1]:
import os
from gai.lib.constants import DEFAULT_GUID
from gai.asm.agents import ToolUseAgent
from gai.mcp.client.mcp_client import McpAggregatedClient
from gai.lib.config import config_helper
from gai.messages import FileMonologue
from gai.lib.tests import make_local_tmp

# Configure Monologue

here = make_local_tmp()
file_path = os.path.join(here, "monologue.json")
monologue = FileMonologue(agent_name="ToolUseAgent",file_path=file_path)

# Configure MCP

aggregated_client = McpAggregatedClient(["mcp-pseudo","mcp-time"])
tools = await aggregated_client.list_tools()

# Configure LLM

llm_config = config_helper.get_client_config(
    {
        "client_type": "anthropic",
        "model": "claude-sonnet-4-20250514",
        "extra": {
            "max_tokens": 32000,
            "temperature": 0.7,
            "top_p": 0.95,
            "tools": True,
            "stream": True,
        },
    }
)


def print_chunk(chunk):
    """
    Helper function to print the chunk of data received from the agent.
    """
    if chunk:
        if isinstance(chunk, str):
            print(chunk, end="", flush=True)
        else:
            if isinstance(chunk, list):
                for item in chunk:
                    if item.get("name"):
                        print(f'Tool: "{item["name"]}"')
                    if item.get("input"):
                        inputs = item.get("input")
                        if isinstance(inputs, dict):
                            for key, value in inputs.items():
                                if isinstance(value, str):
                                    if len(value) > 100:
                                        print(
                                            f"\tInput: {key} = {value[:100]}... (truncated)"
                                        )
                                    else:
                                        print(f"\tInput: {key} = {value}")

---

## Scenario 1: Good Flow

Let us construct a hypothetical conversation below.

In [3]:
from gai.messages import FileDialogue, MessagePydantic

# Create an artificial dialogue history for testing

messages = [
    MessagePydantic(
        **{
            "id": "b1e5f98c-f6eb-47de-a6e2-387510d970f9",
            "header": {
                "sender": "User",
                "recipient": "Sara",
                "timestamp": 1751308157.270983,
                "order": 0,
            },
            "body": {
                "type": "chat.send",
                "dialogue_id": "00000000-0000-0000-0000-000000000000",
                "round_no": 0,
                "step_no": 0,
                "role": "user",
                "content": "It is a very nice weather in Singapore right now.",
            },
        }
    ),
    MessagePydantic(
        **{
            "id": "abbc7961-45dc-4973-aaf4-a6224ed35d37",
            "header": {
                "sender": "Sara",
                "recipient": "User",
                "timestamp": 1751308167.3488164,
                "order": 1,
            },
            "body": {
                "type": "chat.reply",
                "dialogue_id": "00000000-0000-0000-0000-000000000000",
                "round_no": 0,
                "step_no": 1,
                "chunk_no": 10,
                "chunk": "<eom>",
                "role": "assistant",
                "content": "Yes, it is! The weather in Singapore is typically warm and humid, with occasional rain showers. It's a great time to enjoy outdoor activities or relax indoors with a cool drink. How can I assist you today?",
            },
        }
    ),
]
file_path = os.path.join(here, f"{DEFAULT_GUID}.json")
dialogue = FileDialogue(messages=messages, file_path=file_path)
recap = dialogue.extract_recap()


### a) Start Agent

Extract the conversation as recap from dialogue and pass it to the agent. Agent can infer the timezone from the recap.

In [4]:
agent = ToolUseAgent(
    agent_name="ToolUseAgent",
    llm_config=llm_config,
    aggregated_client=aggregated_client,
    monologue=monologue
)

user_message = "What is the current time here?"

resp = agent.start(user_message=user_message, recap=recap)
async for chunk in resp:
    print_chunk(chunk)


I can help you get the current time! Since you mentioned Singapore in the conversation recap, I'll get the current time in Singapore timezone for you.
Tool: "current_time"
	Input: format = YYYY-MM-DD HH:mm:ss
	Input: timezone = Asia/Singapore


### b) Tool Call

Call the MCP and return the result. Also note that the agent is stateless, we can use a new instance to continue the conversation.


In [5]:
agent = ToolUseAgent(
    agent_name="ToolUseAgent",
    llm_config=llm_config,
    aggregated_client=aggregated_client,
    monologue=monologue,
)

resp = agent.resume()
async for chunk in resp:
    print_chunk(chunk)
print(f"\ncompleted state: {agent.fsm.state}")
assert agent.fsm.state == "IS_TERMINATE"

The current time in Singapore is **8:47:05 PM on September 15, 2025**. Perfect evening time to enjoy that nice weather you mentioned!

completed state: IS_TERMINATE


### c) End Conversation

Once the agent has finished its task, calling resume() will fail. To continue, you can use resume(user_message) to update the conversation or start a new one with start(user_message).

In [6]:
from gai.asm.agents.tool_use_agent import AutoResumeError
try:
    agent = ToolUseAgent(
        agent_name="ToolUseAgent",
        llm_config=llm_config,
        aggregated_client=aggregated_client,
        monologue=monologue,
    )
    resp = agent.resume()
    async for chunk in resp:
        print_chunk(chunk)
except AutoResumeError as e:
    print(f"Test passed: conversation is over. {str(e)}")

    # Save the user message and assistant message to the dialogue at end of the conversation
    assistant_message = agent.fsm.state_bag["assistant_message"]
    dialogue.add_user_message(recipient="Sara", content=user_message)
    dialogue.add_assistant_message(sender="Sara", chunk="<eom>", content=assistant_message)

Test passed: conversation is over. ToolUseAgent.resume: Cannot resume() as agent has completed its task. Either resume(user_message) to update the task or start a new task with start(user_message).


### d) Update conversation

In [7]:
resp = agent.resume("Tell me a one sentence story.")
async for chunk in resp:
    print_chunk(chunk)

As the evening rain began to fall gently on the bustling streets of Singapore at 8:47 PM, Maya discovered a mysterious letter tucked beneath her apartment door that would change her life forever.


### e) Show monologue

See the monologue for details at `tmp/monologue.json`

In [8]:
import json
from gai.messages import message_helper

# Show the monologue
print("\n───────────────────────── MONOLOGUE START ─────────────────────────")
messages = agent.monologue.list_chat_messages()
for message in messages:
    print(json.dumps(message, indent=4))
print("───────────────────────── MONOLOGUE END ─────────────────────────\n")

# Print memory size
mem_size=message_helper.get_messages_length(messages)
print("Total char size=", mem_size)


───────────────────────── MONOLOGUE START ─────────────────────────
{
    "role": "user",
    "content": "\n            Your name is ToolUseAgent within the context of this conversation and you will always respond as such.\n            Do not refer to yourself as an AI or a bot or confuse your name with other agents.\n           \n            You may respond to my following message using the context you have learnt.\n            \n            What is the current time here?\n\n            Here is a recap of the conversation. Note that the recap may include agents other than yourself.\n            Do not confuse your identity and do not mention the recap. Just continue.\n            User: It is a very nice weather in Singapore right now.\nSara: Yes, it is! The weather in Singapore is typically warm and humid, with occasional rain showers. It's a great time to enjoy outdoor activities or relax indoors with a cool drink. How can I assist you today?\n            \n            \n           

### f) Show dialogue

In [9]:
for msg in dialogue.list_messages():
    print(f"{msg.header.sender}: {msg.body.content}")

User: It is a very nice weather in Singapore right now.
Sara: Yes, it is! The weather in Singapore is typically warm and humid, with occasional rain showers. It's a great time to enjoy outdoor activities or relax indoors with a cool drink. How can I assist you today?
User: Sara, What is the current time here?
Sara: The current time in Singapore is **8:47:05 PM on September 15, 2025**. Perfect evening time to enjoy that nice weather you mentioned!


---

## Scenario 2: Agent interrupt user for input

We will invite the agent to ask questions in this scenario.

### a) Start Agent

In [7]:
agent = ToolUseAgent(
    agent_name="ToolUseAgent",
    llm_config=llm_config,
    aggregated_client=aggregated_client,
    monologue=monologue,
)

We know the agent is expecting response from the user when it calls "user_input" tool.

In [8]:
user_message = "What is the current time? Please ask if you need more information."

resp = agent.start(user_message=user_message)
async for chunk in resp:
    print_chunk(chunk)


I'm ToolUseAgent, and I'd be happy to help you get the current time. However, I need a bit more information to provide you with the most useful response.
Tool: "user_input"


### b) This will throw error unless user input is provided

In [9]:
from gai.asm.agents.tool_use_agent import PendingUserInputError
try:
    resp = agent.resume()
    async for chunk in resp:
        print_chunk(chunk)
except PendingUserInputError as e:
    assert "pending user input" in str(e)
    print(f"Error occurred: {str(e)}")
    print("This is expected as the agent is waiting for user input.")


ERROR    ToolUserAgent.resume: Pending user input cannot proceed.

Exception: tool_user_agent.resume: Error while handling PendingUserInputError. Cannot fast forward to IS_TERMINATE.

### c) Can resume normally after user input

In [10]:
# IS_TOOL_CALL -> TOOL_USE
resp = agent.resume("Use SGT")
async for chunk in resp:
    print_chunk(chunk)



Tool: "current_time"
	Input: format = YYYY-MM-DD HH:mm:ss
	Input: timezone = Asia/Singapore


### d) Resume the rest of the conversation

In [11]:
from gai.asm.agents.tool_use_agent import AutoResumeError
while True:
    try:
        resp = agent.resume()
        async for chunk in resp:
            print_chunk(chunk)
    except AutoResumeError as e:
        print("\nTest passed: conversation is over.")
        break
    except Exception as e:        
        print("Unexpected error:", str(e))
        raise

The current time in Singapore (SGT) is **2025-09-15 20:55:56** (8:55:56 PM).

Test passed: conversation is over. ToolUseAgent.resume: Cannot resume() as agent has completed its task. Either resume(user_message) to update the task or start a new task with start(user_message).


---

## Scenario 3: User interrupt agent with adhoc input

In this scenario, the user tries to distract the agent by interrupting it with an adhoc input.

### a) Start Agent


In [12]:
agent = ToolUseAgent(
    agent_name="ToolUseAgent",
    llm_config=llm_config,
    aggregated_client=aggregated_client,
    monologue=monologue,
)

# Start the agent

user_message = "What is the current time? Please ask if you need more information."

resp = agent.start(user_message=user_message)
async for chunk in resp:
    print_chunk(chunk)

I'll get the current time for you. To provide the most useful format, let me ask what you prefer.
Tool: "user_input"


### b) User interrupt agent

Should be able to interrupt the agent and continue with original task.

In [13]:
resp = agent.resume("Tell me a one paragraph joke")
async for chunk in resp:
    print_chunk(chunk)


I need to get the current time for you. Let me use the default format:
Tool: "current_time"
	Input: format = YYYY-MM-DD HH:mm:ss


### c) Resume with input

Provide required response and continue with original task.

In [14]:
resp = agent.resume("Use SGT")
async for chunk in resp:
    print_chunk(chunk)


The current time is **2025-09-15 12:56:58** (UTC).

If you need the time in a different timezone or format, just let me know!


---

## Scenario 4: Undo last state

In this scenario, the user undo the first message and provide a new one.

### a) Start Agent


In [16]:
agent = ToolUseAgent(
    agent_name="ToolUseAgent",
    llm_config=llm_config,
    aggregated_client=aggregated_client,
    monologue=monologue,
)

# Start the agent

user_message = "What is the current time? Please ask if you need more information."

resp = agent.start(user_message=user_message)
async for chunk in resp:
    print_chunk(chunk)

I'll get the current time for you. To provide the most accurate information, I need to know what format and timezone you'd prefer.
Tool: "user_input"


In [13]:
state =  agent.undo()
print(f"current state is {state}")

current state is IS_TOOL_CALL


In [29]:
resp = agent.resume("Tell me a one paragraph joke")
async for chunk in resp:
    print_chunk(chunk)


Here's a one paragraph joke for you:

A man walks into a library and asks for books on paranoia. The librarian whispers, "They're right behind you!" The man spins around frantically, but sees nothing except normal bookshelves. He turns back to the librarian, who points to a section labeled "Psychology" and says, "No seriously, the paranoia books are literally right behind you on that shelf." The man sheepishly walks over, grabs a book, and mutters, "I knew they were watching me... I just didn't know they were so well-organized about it."

Hope that gave you a chuckle! Is there anything else I can help you with today?


Confirm by checking the monologue.

In [30]:
agent.monologue.list_chat_messages()

[{'role': 'user',
  'content': '\n            Your name is ToolUseAgent within the context of this conversation and you will always respond as such.\n            Do not refer to yourself as an AI or a bot or confuse your name with other agents.\n           \n            You may respond to my following message using the context you have learnt.\n            Tell me a one paragraph joke\n            \n            You may ask me for more information if you need to clarify my request but ask just enough to get the information you need to get started.\n            If you need to ask for more information, please use the "user_input" tool to get the information from me.\n            Never call "user_input" without providing a description of what you need from me.\n            Do not be vague and expect an input from me, be specific about what you want.\n            '},
 {'role': 'assistant',
  'content': [{'citations': None,
    'text': 'Here\'s a one paragraph joke for you:\n\nA man walks in